## Part 5: Inference function

`predict_podium(record)` answers one question: will this driver finish on the podium (top 3) in this race? It takes the pre-race features of **one driver in one race** and returns the podium probability and a yes/no answer. It uses the final model from Part 4, so nothing is retrained.

The function follows the same pipeline as the training data: the model was trained on 17 pre-race features (`FEATURES`, Part 3), so a record is only accepted if **all 17 are present**. If any feature is missing, the function does **not** make a prediction. It says which features are missing instead.

In [ ]:
# LOCAL SETUP (dropped when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import POST_RACE_COLS
from IPython.utils.capture import capture_output
with capture_output():
    %run p4_models.ipynb

### 5.1 The function

* **Input:** a dictionary with the features in `FEATURES` (Part 3). A feature that is unknown can be left out or set to `None` / `NaN`.
* **Step 1:** refuse the record (error) if it contains a post-race column (`positionOrder`, `points`, `status`, ...), because those are not known before the race.
* **Step 2:** check the 17 features one by one. A feature is **missing** if it is absent, `None` or `NaN`. A feature is **invalid** if it is not a finite number (the model only takes numbers).
* **Step 3, if anything is missing or invalid:** no prediction. The function returns `status = "refused"`, no probability, the list of missing features (with their feature group from Part 3) and the list of invalid features.
* **Step 4, if all 17 features are valid:** build one row with the features in the same order as in training and give it to `FINAL_MODEL`. The model already contains the preprocessing from Part 4 (scaling), so it is not repeated here.
* **Output when it predicts:** `status = "ok"`, the podium probability and the yes/no answer (probability at least `FINAL_THRESHOLD`).

**Why refuse instead of filling the gap?** The Part 4 pipeline can fill a missing value with the training median, but a filled value is not the real value. For a feature such as `grid` or `qual_position` the result would look like a normal prediction while being based on an invented input. Refusing and naming the missing features makes the problem visible.

In [ ]:
import numpy as np
import pandas as pd

GROUP_OF = {f: g for g, cols in FEATURE_GROUPS.items() for f in cols}   # feature -> feature group (Part 3)


def _is_missing(v):
    """True for None, NaN, pd.NA (a value that was not given)."""
    try:
        return bool(pd.isna(v))
    except (TypeError, ValueError):                  # e.g. a list: not a missing value, but not a number either
        return False


def predict_podium(record, model=None, threshold=None):
    model = FINAL_MODEL if model is None else model
    threshold = FINAL_THRESHOLD if threshold is None else threshold

    # 1) post-race information is never allowed
    leaky = set(record.keys()) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:
        raise ValueError(f"post-race information is not allowed: {sorted(leaky)}")

    # 2) every feature of the training pipeline must be present and a finite number
    values, missing, invalid = {}, [], []
    for f in FEATURES:
        v = record.get(f)
        if _is_missing(v):
            missing.append(f)
            continue
        try:
            x = float(v)
        except (TypeError, ValueError):
            invalid.append(f)
            continue
        if not np.isfinite(x):
            invalid.append(f)
            continue
        values[f] = x

    # 3) something is missing or invalid -> no prediction, say what is wrong
    if missing or invalid:
        problems = []
        if missing:
            problems.append(f"{len(missing)} feature(s) missing: " + ", ".join(f"{f} ({GROUP_OF[f]})" for f in missing))
        if invalid:
            problems.append(f"{len(invalid)} feature(s) not a finite number: " + ", ".join(invalid))
        return {
            "status": "refused",
            "podium_probability": None,
            "podium_prediction": None,
            "missing_features": missing,
            "invalid_features": invalid,
            "message": "No prediction made. " + "; ".join(problems) + ".",
        }

    # 4) complete record -> predict with the final model
    X = pd.DataFrame([values], columns=FEATURES)
    proba = float(model.predict_proba(X)[0, 1])
    return {
        "status": "ok",
        "podium_probability": proba,
        "podium_prediction": int(proba >= threshold),
        "missing_features": [],
        "invalid_features": [],
        "message": f"Prediction made: all {len(FEATURES)} features present.",
    }

### 5.2 Example A: complete record
A real driver-race from the validation years: the first one where all features are known. The finishing position is only printed to describe the example, it is not an input.

In [ ]:
def describe_row(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    finish = outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"]
    return f"{driver}, {race} {row['year']} (actual finish: P{int(finish)})"

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()

print(describe_row(row_A))
print("input:", record_A)
print("output:", predict_podium(record_A))

All 17 features are given, so the function predicts. The output has the podium probability and the yes/no answer at the validation threshold. The function running without errors does not show that the prediction is right: one record proves nothing about accuracy. The accuracy of the model is measured in Part 4 on the validation and test sets.

### 5.3 Example B: partially missing record
A real driver-race where information is missing: the first validation row of a driver's first Grand Prix. He has no earlier races, so there is no form, and the row also has gaps in the standings. The unknown fields are left out of the dictionary, as they would be when the record is built from what is known before the race.

The function does not predict. It lists the missing features.

In [ ]:
first_race_rows = table[(table["split"] == "validation") & table["driver_form5"].isna()]
row_B = first_race_rows.iloc[0]
record_B = {k: v for k, v in row_B[FEATURES].to_dict().items() if pd.notna(v)}

print(describe_row(row_B))
print("input:", record_B)
result_B = predict_podium(record_B)
print("output:", result_B)
print(result_B["message"])

### 5.4 Checks
* For validation rows with all 17 features, the function gives the same probability as calling the model directly.
* For validation rows with a missing value, the function gives no probability, and the missing features it lists are exactly the empty columns of that row.
* It still refuses post-race columns.

In [ ]:
val = table[table["split"] == "validation"]
is_complete = val[FEATURES].notna().all(axis=1)

# complete rows: same probability as the model
val_ok = val[is_complete]
direct = FINAL_MODEL.predict_proba(val_ok[FEATURES])[:, 1]
results_ok = [predict_podium(r[FEATURES].to_dict()) for _, r in val_ok.iterrows()]
one_by_one = np.array([r["podium_probability"] for r in results_ok])
print("complete validation rows:", len(val_ok), "| largest difference to the model:", abs(direct - one_by_one).max())
assert all(r["status"] == "ok" for r in results_ok)
assert np.allclose(direct, one_by_one)

# incomplete rows: no prediction, and the list of missing features is exactly the empty columns
val_bad = val[~is_complete]
results_bad = [predict_podium(r[FEATURES].to_dict()) for _, r in val_bad.iterrows()]
assert all(r["status"] == "refused" and r["podium_probability"] is None for r in results_bad)
for (_, r), res in zip(val_bad.iterrows(), results_bad):
    assert res["missing_features"] == [f for f in FEATURES if pd.isna(r[f])]
print("incomplete validation rows:", len(val_bad), "| all refused, the missing features match the empty columns")

# an invalid (non-numeric) value is refused too, a post-race column raises an error
assert predict_podium({**record_A, "grid": "front"})["invalid_features"] == ["grid"]
for col in ["positionOrder", "points", "status", "podium"]:
    try:
        predict_podium({**record_A, col: 1})
    except ValueError:
        print("refused (post-race):", col)

### 5.5 How many records are refused, and why
Because every feature must be present, records with structural gaps are refused as well (standings in round 1 of a season, form in a driver's first race, a missing qualifying time). The table shows how many validation rows this affects and which features are missing most often. The test set is not used here.

In [ ]:
share = len(val_bad) / len(val)
print(f"validation rows refused: {len(val_bad)} of {len(val)} ({share:.1%}); predicted: {len(val_ok)}")

missing_counts = pd.Series([f for r in results_bad for f in r["missing_features"]]).value_counts()
display(missing_counts.rename("rows with this feature missing").to_frame().assign(group=lambda d: d.index.map(GROUP_OF)))

### 5.6 Limitations
* The function takes the features computed in Part 3 (form, standings, circuit history). It does not build them from raw race data.
* A record with any missing feature gets no prediction. This includes rows with structural gaps (round 1 of a season, a driver's first race, a missing qualifying time), see 5.5, so for those races the function cannot be used. The model of Part 4 could give a number for them (it fills the gaps with training medians), but that number would rest on invented inputs.
* It checks that the features are present and numeric. It does not check that the values are plausible (for example a `grid` of 50).
* It predicts one driver at a time, so it can say "podium" for more or fewer than three drivers in the same race. It cannot be better than the Part 4 model.